# SLTK: Sea Level Tool Kit — Example Usage

This notebook walks through the core workflow of the library: loading a station's time series, computing its long-term trend and seasonal cycle, finding nearby stations, and comparing trends across multiple stations.

**Before running this notebook**, make sure you have installed the library (`pip install -e .` from the repository root) and that you are using the `sltk` kernel (top right corner of this notebook).

In [ ]:
import sltk

## 1. Load a single station's cleaned time series

`load_station_series` reads a raw PSMSL `.rlrdata.txt` file, converts missing-value codes (`-99999`) to `NaN`, and returns a clean time-indexed DataFrame.

In [ ]:
trieste = sltk.load_station_series("data/stations/154.rlrdata.txt")
trieste.head()

## 2. Compute the long-term trend and seasonal cycle

`SeaLevelAnalysis` computes the linear trend (mm/year) via linear regression, and the average seasonal cycle (mean sea level per calendar month).

In [ ]:
analysis = sltk.SeaLevelAnalysis(trieste["sea_level_mm"])

trend = analysis.compute_trend()
print(trend)  # e.g. {'trend_mm_per_year': 1.35, 'r_squared': 0.29, 'p_value': 4e-124}

In [ ]:
seasonal = analysis.seasonal_cycle()
seasonal

## 3. Load station metadata and find the nearest stations

`StationLoader` reads station metadata (name, ID, coordinates) into a `GeoDataFrame`. `Distances` computes Euclidean or geodetic (Haversine) distances between stations, and can find the nearest stations to a given one.

In [ ]:
loader_metadata = sltk.StationLoader("data/metadata_stations.csv")

dist = sltk.Distances(loader_metadata.metadata, id_col="ID")
nearest = dist.nearest(station_id=154, n=3, method="geodetic")
nearest[["Station Name", "distance"]]

## 4. Compare trends across multiple stations

`compare_stations` computes the trend for each station in the given list and merges the results with their geographic location, returning a single `GeoDataFrame` ready for mapping or further analysis.

In [ ]:
station_ids = [154, 2098, 2075, 2095, 2142, 2093, 2090]
comparison = sltk.compare_stations(loader_metadata, station_ids, data_dir="data/stations")
comparison[["Station Name", "trend_mm_per_year", "r_squared", "p_value"]]

## 5. (Optional) Map the trends by station

A simple visualization of sea level trends across the selected stations, colored by trend magnitude.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
comparison.plot(column="trend_mm_per_year", ax=ax, legend=True,
                 cmap="coolwarm", markersize=100, edgecolor="black",
                 legend_kwds={"label": "Trend (mm/year)"})

for _, row in comparison.iterrows():
    ax.annotate(str(row["ID"]), xy=(row.geometry.x, row.geometry.y),
                xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_title("Sea level trend by station (mm/year)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()